# Physics-Informed Neural Network for the Viscous Burgers Equation

This notebook demonstrates a structured PINN implementation for the 1-D viscous Burgers equation:

$$
\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}, \quad x \in [-1, 1],\; t \in [0, 1]
$$

with initial condition $u(x, 0) = -\sin(\pi x)$ and Dirichlet boundaries $u(\pm 1, t) = 0$.

**Architecture highlights:**
- Random Fourier feature input encoding (Tancik et al. 2020) to overcome spectral bias
- Residual-connected MLP blocks (Wang et al. 2021) for stable deep training
- Adaptive (learnable) per-term loss weighting
- Two-phase optimisation: Adam with cosine LR schedule → L-BFGS fine-tuning
- Latin Hypercube collocation sampling for efficient domain coverage

The analytical reference is computed via the **Cole-Hopf transform**.


## 1 — Setup and Configuration

All configuration is Pydantic-validated.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from physics_informed_neural_network.config import ProjectConfig
from physics_informed_neural_network.data import evaluate_reference_solution, generate_reference_solution, sample_observations
from physics_informed_neural_network.pipeline import run_experiment
from physics_informed_neural_network.plotting import (
    apply_plot_style,
    plot_comparison,
    plot_loss_history,
    plot_pointwise_error,
    plot_reference_solution,
    plot_residual_distribution,
    plot_time_slices,
)

apply_plot_style()
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

In [ ]:
config = ProjectConfig()
config.artifacts.output_dir = PROJECT_ROOT / "artifacts" / "notebook_burgers_pinn"
display(config.model_dump())

## 2 — Analytical Reference Solution (Cole-Hopf Transform)

High-fidelity reference on a $256 \times 100$ grid.

In [ ]:
reference = generate_reference_solution(config.pde, config.data)
print(f"Grid: {reference.nx} × {reference.nt} = {reference.nx * reference.nt:,} points")
print(f"ν = {reference.viscosity:.6f}")

In [ ]:
fig_ref = plot_reference_solution(reference)
fig_ref

## 3 — Train the PINN

In [ ]:
experiment = run_experiment(config)

## 4 — Results: PINN vs Reference

In [ ]:
u_ref = evaluate_reference_solution(experiment.x_pred, experiment.t_pred, config.pde.viscosity)
fig_cmp = plot_comparison(experiment.x_pred, experiment.t_pred, u_ref, experiment.u_pred)
fig_cmp

In [ ]:
fig_slices = plot_time_slices(experiment.x_pred, experiment.t_pred, u_ref, experiment.u_pred)
fig_slices

## 5 — Error Analysis

In [ ]:
metrics_df = pd.DataFrame([experiment.metrics.model_dump()])
display(metrics_df)

In [ ]:
fig_err = plot_pointwise_error(experiment.x_pred, experiment.t_pred, u_ref, experiment.u_pred)
fig_err

## 6 — Training Diagnostics

In [ ]:
fig_loss = plot_loss_history(experiment.history)
fig_loss

In [ ]:
display(experiment.history.to_frame().tail(5))

## 7 — PDE Residual Analysis

How well does the trained PINN satisfy the Burgers equation at interior points?

We sample 5,000 fresh interior collocation points and evaluate the PDE residual $r = u_t + u \cdot u_x - \nu\, u_{xx}$ using automatic differentiation through the trained model.


In [ ]:
import torch
from physics_informed_neural_network.data import sample_interior_collocation
from physics_informed_neural_network.physics import BurgersResidual

# Sample fresh interior collocation points
eval_pts = sample_interior_collocation(config.pde, 5000, seed=999)
device = next(experiment.trainer.model.parameters()).device
eval_tensor = torch.tensor(eval_pts, dtype=torch.float32, device=device).requires_grad_(True)

# Compute PDE residuals
pde = BurgersResidual(viscosity=config.pde.viscosity)
with torch.enable_grad():
    residuals = pde.residual(experiment.trainer.model, eval_tensor).detach().cpu().numpy()

print(f"PDE residual statistics (n={len(residuals)}):")
print(f"  Mean |r|:  {np.mean(np.abs(residuals)):.6e}")
print(f"  Max  |r|:  {np.max(np.abs(residuals)):.6e}")
print(f"  Std  r:    {np.std(residuals):.6e}")

fig_res = plot_residual_distribution(residuals)
fig_res

## 8 — Summary and Artifacts

In [ ]:
display(experiment.summary.model_dump())

In [ ]:
artifact_df = pd.DataFrame([
    {"artifact": k, "path": str(v)} for k, v in experiment.artifact_paths.items()
])
display(artifact_df)

## 9 — Interpretation

### What this notebook demonstrates

- **Physics-informed learning**: the network is trained to satisfy the Burgers PDE, boundary conditions, initial conditions, and sparse observations simultaneously
- **Cole-Hopf reference**: the exact analytical solution provides a rigorous ground truth
- **Two-phase optimization**: Adam for broad exploration, L-BFGS for fine-tuning near convergence
- **Fourier features**: random Fourier feature encoding helps the network represent high-frequency structure

### Key metrics to watch

| Metric | Good | Excellent |
|--------|------|-----------|
| Relative $L^2$ error | $< 5\%$ | $< 1\%$ |
| Max absolute error | $< 0.1$ | $< 0.01$ |
| PDE residual (mean) | $< 10^{-3}$ | $< 10^{-4}$ |
